# GCP Vertex AI TPM Quota Consumption Verification Hands-on

## Purpose

This notebook is a hands-on test to **directly verify how TPM (Tokens Per Minute) quota is actually consumed** in the GCP Vertex AI environment.

In particular, it verifies **whether cached tokens are deducted from the TPM quota at a 1:1 ratio without any discount**.

### How to Use the Verification Results

The precise token count structure from `usageMetadata` confirmed through this test serves as the core basis for designing quota limitation logic.

### Test Flow

| Step | Description | Method |
|------|-------------|--------|
| **Step 1** | Create 12k cache and run multi-call test | Python code execution |
| **Step 2** | Check TPM consumption graph in GCP Console | Console monitoring |

---
## Prerequisites

### Install Required Packages

Run the cell below if the Vertex AI SDK is not installed.

In [ ]:
!pip install --upgrade google-cloud-aiplatform google-cloud-monitoring

### GCP Authentication

| Environment | Authentication Method |
|-------------|----------------------|
| **Colab Enterprise** | No separate authentication needed (the runner's IAM permissions are automatically applied) |
| **Vertex AI Workbench / Cloud Shell** | No separate authentication needed |
| **Google Colab (free)** | Uncomment and run `auth.authenticate_user()` in the cell below |
| **Local environment** | Uncomment and run `gcloud auth` in the cell below |

> **Note:** Colab Enterprise runs within a GCP project, so the user's IAM permissions are automatically applied.  
> However, the runner must have the **Vertex AI User** role (or equivalent permissions) assigned.

In [ ]:
# ──────────────────────────────────────────────────
# Colab Enterprise / Workbench / Cloud Shell → Skip this cell
# ──────────────────────────────────────────────────

# If running on Google Colab (free):
# from google.colab import auth
# auth.authenticate_user()

# If running on a local environment:
# !gcloud auth application-default login

---
## Step 1: Create 12k Cache and Run Multi-Call Test via Python Code

We call the **Vertex AI SDK directly** without going through a gateway to generate pure baseline metrics.

### 1-1. Environment Variable Configuration

> **Make sure to change `PROJECT_ID` below to your own GCP project ID.**

In [ ]:
PROJECT_ID = "YOUR_PROJECT_ID"  # ← Change to your project ID
LOCATION = "us-central1"
MODEL_ID = "gemini-2.5-flash"  # Gemini 2.5 Flash

# Test parameters
NUM_CALLS = 10           # Number of API calls
CACHE_TTL_HOURS = 1      # Cache retention time in hours
SYSTEM_PROMPT_REPEAT = 500  # System prompt repetition count (to generate ~10k-12k tokens)

print(f"Project: {PROJECT_ID}")
print(f"Region: {LOCATION}")
print(f"Model: {MODEL_ID}")
print(f"Number of calls: {NUM_CALLS}")

### 1-2. Initialize Vertex AI

In [ ]:
import datetime
import time

import vertexai
from vertexai.generative_models import GenerativeModel
from vertexai.preview import caching

vertexai.init(project=PROJECT_ID, location=LOCATION)
print("Vertex AI initialization complete")

### 1-2a. Cloud Monitoring Metric Query Setup

Use the Google Cloud Monitoring API to programmatically query Vertex AI token-related metrics.

| Metric | Description |
|--------|-------------|
| `token_count` | Number of tokens used in online serving |
| `consumed_token_throughput` | Actual token throughput reflecting burndown rate (TPM calculation possible) |

In [ ]:
from google.cloud import monitoring_v3
from google.api_core import client_options as client_options_lib
from datetime import timedelta

METRIC_TOKEN_COUNT = "aiplatform.googleapis.com/publisher/online_serving/token_count"
METRIC_CONSUMED_THROUGHPUT = "aiplatform.googleapis.com/publisher/online_serving/consumed_token_throughput"

monitoring_client = monitoring_v3.MetricServiceClient(
    client_options=client_options_lib.ClientOptions(
        quota_project_id=PROJECT_ID,
    ),
)


def query_metric(metric_type, start_time, end_time, alignment_seconds=60,
                 aligner=monitoring_v3.Aggregation.Aligner.ALIGN_DELTA):
    """Query Vertex AI metrics from Cloud Monitoring."""
    project_name = f"projects/{PROJECT_ID}"

    interval = monitoring_v3.TimeInterval(
        start_time=start_time,
        end_time=end_time,
    )

    aggregation = monitoring_v3.Aggregation(
        alignment_period={"seconds": alignment_seconds},
        per_series_aligner=aligner,
    )

    filter_str = (
        f'metric.type = "{metric_type}"'
        f' AND resource.labels.model_user_id = "{MODEL_ID}"'
        f' AND resource.labels.location = "{LOCATION}"'
    )

    try:
        return list(monitoring_client.list_time_series(
            request={
                "name": project_name,
                "filter": filter_str,
                "interval": interval,
                "aggregation": aggregation,
                "view": monitoring_v3.ListTimeSeriesRequest.TimeSeriesView.FULL,
            }
        ))
    except Exception as e:
        print(f"  Query error: {e}")
        return []


def get_point_value(point):
    """Extract numeric value from TypedValue."""
    v = point.value
    return v.double_value if v.double_value != 0 else v.int64_value


def format_ts(t):
    """Convert timestamp to string."""
    if hasattr(t, 'strftime'):
        return t.strftime('%H:%M:%S')
    elif hasattr(t, 'seconds'):
        dt_obj = datetime.datetime.fromtimestamp(t.seconds, tz=datetime.timezone.utc)
        return dt_obj.strftime('%H:%M:%S')
    return str(t)


def display_metric(time_series_list, label=""):
    """Display metric time series data."""
    if not time_series_list:
        print(f"  [{label}] No data")
        print(f"    -> Metrics may take 2-5 minutes to reflect. Please retry shortly.")
        return []

    all_values = []
    for ts in time_series_list:
        labels = dict(ts.metric.labels)
        print(f"\n  [{label}] Metric labels: {labels}")
        print(f"  Data points: {len(ts.points)}")

        for pt in sorted(ts.points, key=lambda p: p.interval.end_time):
            val = get_point_value(pt)
            t = pt.interval.end_time
            print(f"    {format_ts(t)} UTC -> {val:>12,.2f}")
            all_values.append({"time": t, "value": val, "labels": labels})

    return all_values


def fetch_run_metrics(start_time, end_time, run_label):
    """Query token_count and consumed_token_throughput for a given start/end time window."""
    print(f"\n{'\u2501' * 70}")
    print(f"  Cloud Monitoring Metric Query - {run_label}")
    print(f"  Test start: {start_time.strftime('%Y-%m-%d %H:%M:%S')} UTC")
    print(f"  Test end: {end_time.strftime('%Y-%m-%d %H:%M:%S')} UTC")
    print(f"{'\u2501' * 70}")

    metrics = [
        ("token_count", METRIC_TOKEN_COUNT),
        ("consumed_token_throughput", METRIC_CONSUMED_THROUGHPUT),
    ]

    results = {}

    for name, mtype in metrics:
        print(f"\n{'\u2500' * 70}")
        print(f"  {name}")
        print(f"  ({mtype})")
        print(f"{'\u2500' * 70}")

        # Start window (2 minutes before/after)
        print(f"\n  [Start window] {start_time.strftime('%H:%M:%S')} UTC +/- 2 min")
        s_data = query_metric(
            mtype,
            start_time - timedelta(minutes=2),
            start_time + timedelta(minutes=2),
        )
        s_vals = display_metric(s_data, f"{name} (start)")

        # End window (2 minutes before/after)
        print(f"\n  [End window] {end_time.strftime('%H:%M:%S')} UTC +/- 2 min")
        e_data = query_metric(
            mtype,
            end_time - timedelta(minutes=2),
            end_time + timedelta(minutes=2),
        )
        e_vals = display_metric(e_data, f"{name} (end)")

        # Full period (1 min before start ~ 5 min after end)
        print(f"\n  [Full period] {start_time.strftime('%H:%M:%S')} ~ {end_time.strftime('%H:%M:%S')} UTC (+buffer)")
        f_data = query_metric(
            mtype,
            start_time - timedelta(minutes=1),
            end_time + timedelta(minutes=5),
        )
        f_vals = display_metric(f_data, f"{name} (full)")

        results[name] = {
            "start": s_vals,
            "end": e_vals,
            "full": f_vals,
            "raw": f_data,
        }

    return results


print("Cloud Monitoring helper functions defined")
print(f"  Filter: model_user_id={MODEL_ID}, location={LOCATION}")

In [ ]:
print("Exploring available Vertex AI publisher metrics in the project...\n")

# 1. Check metric descriptors
for metric_type in [METRIC_TOKEN_COUNT, METRIC_CONSUMED_THROUGHPUT]:
    try:
        descriptor = monitoring_client.get_metric_descriptor(
            name=f"projects/{PROJECT_ID}/metricDescriptors/{metric_type}"
        )
        kind = getattr(descriptor.metric_kind, 'name', str(descriptor.metric_kind))
        vtype = getattr(descriptor.value_type, 'name', str(descriptor.value_type))
        print(f"[OK] {descriptor.display_name}")
        print(f"  Type: {descriptor.type}")
        print(f"  Kind: {kind}, Value: {vtype}")
        print(f"  Metric Labels: {[l.key for l in descriptor.labels]}")
        print()
    except Exception as e:
        print(f"[FAIL] {metric_type}: {e}\n")

# 2. Query last 1 hour data (with gemini filter)
print(f"{'\u2500' * 70}")
print(f"Querying last 1 hour data (gemini-2.5-flash filter)...")
print(f"  model_user_id={MODEL_ID}, location={LOCATION}")
print(f"{'\u2500' * 70}")

end_time = datetime.datetime.now(datetime.timezone.utc)
start_time = end_time - timedelta(hours=1)

for metric_name, metric_type in [("token_count", METRIC_TOKEN_COUNT),
                                  ("consumed_throughput", METRIC_CONSUMED_THROUGHPUT)]:
    # Apply gemini filter
    filter_str = (
        f'metric.type = "{metric_type}"'
        f' AND resource.labels.model_user_id = "{MODEL_ID}"'
        f' AND resource.labels.location = "{LOCATION}"'
    )
    try:
        results = list(monitoring_client.list_time_series(
            request={
                "name": f"projects/{PROJECT_ID}",
                "filter": filter_str,
                "interval": monitoring_v3.TimeInterval(
                    start_time=start_time,
                    end_time=end_time,
                ),
                "view": monitoring_v3.ListTimeSeriesRequest.TimeSeriesView.HEADERS,
            }
        ))
        if results:
            print(f"\n[{metric_name}] {len(results)} series found")
            for ts in results:
                print(f"  Resource Labels: {dict(ts.resource.labels)}")
                print(f"  Metric Labels : {dict(ts.metric.labels)}")
                print()
        else:
            print(f"\n[{metric_name}] No gemini-2.5-flash data (last 1 hour)")
            print(f"  -> Run the test (cells 1-4) first to generate metrics.")
    except Exception as e:
        print(f"\n[{metric_name}] Query failed: {e}")

### 1-3. Create Large System Prompt (Cache)

Create a cache using dummy text of approximately **10,000 to 12,000 tokens** as the system prompt.  
This cache will be reused in all subsequent calls, and **how it is reflected in the TPM quota is the key verification point**.

In [ ]:
print("Creating large system prompt (cache)...")

long_system_instruction = (
    "You are a professional AI assistant that guides internal security policies and architecture. "
    * SYSTEM_PROMPT_REPEAT
)

print(f"System prompt length: {len(long_system_instruction):,} characters")
print(f"Estimated token count: ~{len(long_system_instruction) // 4:,} tokens (estimate)")

In [ ]:
cached_content = caching.CachedContent.create(
    model_name=MODEL_ID,
    system_instruction=long_system_instruction,
    ttl=datetime.timedelta(hours=CACHE_TTL_HOURS),
)

print(f"Cache created successfully!")
print(f"  Cache name: {cached_content.name}")
print(f"  Cache expiry: {CACHE_TTL_HOURS} hour(s)")

### 1-4. 10 Consecutive API Calls Using Cache

Simulate short questions that internal developers would ask, making 10 consecutive calls.  
Token counts are extracted from `usageMetadata` for each call.

#### Verification Points
- `prompt_token_count` (Total Input): Verify that each call shows **approximately 12,000+**
- `cached_content_token_count`: Check if cached tokens are separately reported
- This entire `prompt_token_count` is the **input token count per call** as recognized by Vertex AI

In [ ]:
print(f"Starting {NUM_CALLS} consecutive API calls using cache...\n")
print("=" * 65)

model = GenerativeModel.from_cached_content(cached_content=cached_content)

results = []

cached_start_time = datetime.datetime.now(datetime.timezone.utc)
print(f"  >> Start time (UTC): {cached_start_time.isoformat()}\n")

for i in range(1, NUM_CALLS + 1):
    response = model.generate_content("Summarize cloud security configurations.")

    usage = response.usage_metadata

    result = {
        "call_number": i,
        "prompt_token_count": usage.prompt_token_count,
        "cached_content_token_count": usage.cached_content_token_count,
        "candidates_token_count": usage.candidates_token_count,
    }
    results.append(result)

    print(f"--- [Call {i:2d}/{NUM_CALLS}] ---")
    print(f"  Total Input  (prompt_token_count)          : {usage.prompt_token_count:>8,}")
    print(f"  Cached       (cached_content_token_count)  : {usage.cached_content_token_count:>8,}")
    print(f"  Output       (candidates_token_count)      : {usage.candidates_token_count:>8,}")
    print()

cached_end_time = datetime.datetime.now(datetime.timezone.utc)
print(f"  >> End time (UTC): {cached_end_time.isoformat()}")
print("=" * 65)
print(f"{NUM_CALLS} calls completed!")

### 1-5. Results Summary and Analysis

In [ ]:
total_prompt_tokens = sum(r["prompt_token_count"] for r in results)
total_cached_tokens = sum(r["cached_content_token_count"] for r in results)
total_output_tokens = sum(r["candidates_token_count"] for r in results)
avg_prompt_per_call = total_prompt_tokens / len(results)

print("\u2501" * 65)
print("                    Results Summary")
print("\u2501" * 65)
print(f"  Total number of calls                     : {len(results):>10}")
print(f"  Avg Input tokens per call (prompt_token)  : {avg_prompt_per_call:>10,.0f}")
print(f"  Total Input tokens sum                    : {total_prompt_tokens:>10,}")
print(f"  Total Cached tokens sum                   : {total_cached_tokens:>10,}")
print(f"  Total Output tokens sum                   : {total_output_tokens:>10,}")
print("\u2501" * 65)
print()
print("Expected TPM consumption (Input only):")
print(f"  prompt_token_count x {NUM_CALLS} calls = {total_prompt_tokens:,} tokens")
print()
print("If cache is discounted (only non-cached tokens consumed):")
non_cached_total = total_prompt_tokens - total_cached_tokens
print(f"  (prompt - cached) x {NUM_CALLS} calls = {non_cached_total:,} tokens")
print()
print("\u2501" * 65)
print("If the actual TPM peak in GCP Console quota graph is close to")
print(f"  {total_prompt_tokens:,} -> Cached tokens are deducted 1:1 from TPM")
print(f"  {non_cached_total:,} -> Cached tokens are discounted from TPM")
print("\u2501" * 65)

### 1-6a. Cloud Monitoring Metric Query (With Cache)

Query `token_count` and `consumed_token_throughput` metrics from Cloud Monitoring API for the cached test period.

Each metric is queried across 3 time windows: **start window**, **end window**, and **full period**.

> **Note:** Cloud Monitoring metrics may take **2-5 minutes** to reflect.  
> If no data is found, wait a moment and re-run this cell.

In [ ]:
cached_metrics = fetch_run_metrics(cached_start_time, cached_end_time, "With Cache")

### 1-6. Per-Call Token Usage Visualization

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker

    calls = [r["call_number"] for r in results]
    prompts = [r["prompt_token_count"] for r in results]
    cached = [r["cached_content_token_count"] for r in results]
    outputs = [r["candidates_token_count"] for r in results]

    fig, ax = plt.subplots(figsize=(12, 5))

    bar_width = 0.25
    x = range(len(calls))
    x1 = [i - bar_width for i in x]
    x2 = list(x)
    x3 = [i + bar_width for i in x]

    ax.bar(x1, prompts, width=bar_width, label="Total Input (prompt_token_count)", color="#4285F4")
    ax.bar(x2, cached, width=bar_width, label="Cached (cached_content_token_count)", color="#FBBC04")
    ax.bar(x3, outputs, width=bar_width, label="Output (candidates_token_count)", color="#34A853")

    ax.set_xlabel("Call Number")
    ax.set_ylabel("Token Count")
    ax.set_title("Per-Call Token Usage Comparison")
    ax.set_xticks(list(x))
    ax.set_xticklabels([f"#{c}" for c in calls])
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

except ImportError:
    print("matplotlib is not installed.")
    print("To enable visualization: !pip install matplotlib")

### 1-7. (Control Group) 10 Calls Without Cache Using the Same System Prompt

Without using cache, **send the 12k system prompt directly with each call**.  
This serves as a baseline for comparing TPM consumption between cached and non-cached calls in the GCP Console.

> **Note:** To avoid time overlap with the cache test, it is recommended to wait **approximately 2-3 minutes** before running this cell.

In [ ]:
print(f"(Control Group) Starting {NUM_CALLS} consecutive API calls without cache...\n")
print("=" * 65)

model_no_cache = GenerativeModel(
    model_name=MODEL_ID,
    system_instruction=long_system_instruction,
)

results_no_cache = []

no_cache_start_time = datetime.datetime.now(datetime.timezone.utc)
print(f"  >> Start time (UTC): {no_cache_start_time.isoformat()}\n")

for i in range(1, NUM_CALLS + 1):
    response = model_no_cache.generate_content("Summarize cloud security configurations.")

    usage = response.usage_metadata

    result = {
        "call_number": i,
        "prompt_token_count": usage.prompt_token_count,
        "cached_content_token_count": getattr(usage, "cached_content_token_count", 0) or 0,
        "candidates_token_count": usage.candidates_token_count,
    }
    results_no_cache.append(result)

    print(f"--- [Call {i:2d}/{NUM_CALLS}] ---")
    print(f"  Total Input  (prompt_token_count)          : {usage.prompt_token_count:>8,}")
    print(f"  Cached       (cached_content_token_count)  : {result['cached_content_token_count']:>8,}")
    print(f"  Output       (candidates_token_count)      : {usage.candidates_token_count:>8,}")
    print()

no_cache_end_time = datetime.datetime.now(datetime.timezone.utc)
print(f"  >> End time (UTC): {no_cache_end_time.isoformat()}")
print("=" * 65)
print(f"Non-cached {NUM_CALLS} calls completed!")

### 1-8. Cached vs Non-Cached Results Comparison

In [ ]:
nc_total_prompt = sum(r["prompt_token_count"] for r in results_no_cache)
nc_total_cached = sum(r["cached_content_token_count"] for r in results_no_cache)
nc_total_output = sum(r["candidates_token_count"] for r in results_no_cache)
nc_avg_prompt = nc_total_prompt / len(results_no_cache)

print("\u2501" * 65)
print("         Cached vs Non-Cached Comparison Summary")
print("\u2501" * 65)
print()
print(f"  {'Metric':<35} {'With Cache':>12} {'No Cache':>12}")
print(f"  {'\u2500' * 35} {'\u2500' * 12} {'\u2500' * 12}")
print(f"  {'Avg prompt_token_count per call':<35} {avg_prompt_per_call:>12,.0f} {nc_avg_prompt:>12,.0f}")
print(f"  {'Total Input tokens sum':<35} {total_prompt_tokens:>12,} {nc_total_prompt:>12,}")
print(f"  {'Total Cached tokens sum':<35} {total_cached_tokens:>12,} {nc_total_cached:>12,}")
print(f"  {'Total Output tokens sum':<35} {total_output_tokens:>12,} {nc_total_output:>12,}")
print()
print("\u2501" * 65)
print("Compare the TPM spikes of both tests in the GCP Console.")
print("  If both spikes are similar -> prompt_token_count is fully deducted from TPM regardless of cache")
print("\u2501" * 65)

### 1-8a. Cloud Monitoring Metric Query (Without Cache)

Query `token_count` and `consumed_token_throughput` metrics for the non-cached test period.

> **Note:** Compare the metric values with the cached test to see the impact of caching on TPM consumption.

In [ ]:
no_cache_metrics = fetch_run_metrics(no_cache_start_time, no_cache_end_time, "Without Cache")

### 1-9. (Control Group 2) Preventing Implicit Cache - 10 Calls with Unique System Prompts

Even without explicitly using cache, **repeatedly sending the same system prompt** may cause the Gemini model to apply **implicit caching (automatic caching)** on the server side.

To prevent this, we generate and send a **completely different system prompt for each call**.  
Each prompt includes a unique UUID, timestamp, and random elements, sized at approximately 12,000 tokens.

| Comparison | No Cache (1-7) | Implicit Cache Prevention (1-9) |
|-----------|----------------|----------------------------------|
| System prompt | Same every time | Unique per call |
| Implicit cache possibility | Yes | No |
| Prompt size | ~12,000 tokens | ~12,000 tokens |

> **Note:** Wait **approximately 2-3 minutes** to avoid time overlap with the non-cached test (1-7).

In [ ]:
import random
import string
import uuid

print("(Control Group 2) Preventing Implicit Cache - 10 calls with unique system prompts\n")
print("=" * 65)

POLICY_TOPICS = [
    "The following detailed guidelines are established for the implementation and operation of the Information Security Management System (ISMS). All employees must be familiar with and comply with these guidelines, and violations will be handled according to internal disciplinary regulations.",
    "The Zero Trust model is adopted as the design principle for cloud infrastructure security architecture. All network requests must undergo authentication and authorization regardless of their origin.",
    "According to the data governance framework, all data assets must be assigned a classification level at the time of creation. Classification levels are divided into four tiers: Public, Internal, Confidential, and Top Secret.",
    "Access control policies are designed based on the principle of least privilege. Each user is granted only the minimum permissions necessary for their duties, and permission appropriateness is reviewed quarterly.",
    "Incident response procedures consist of six phases: Detection, Analysis, Containment, Eradication, Recovery, and Post-Incident Review. Responsible parties and escalation paths must be clearly defined for each phase.",
    "Encryption standards apply to both data in transit and data at rest. TLS 1.3 or higher must be used, and AES-256-GCM is adopted as the default symmetric encryption algorithm.",
    "According to API security guidelines, all externally exposed APIs must implement OAuth 2.0 or API key authentication. Rate limiting, input validation, and output encoding must be mandatorily implemented.",
    "As a container security policy, all container images must be built from approved base images. Image scanning is integrated into the CI/CD pipeline for automatic vulnerability checking.",
    "According to network segmentation rules, production, staging, and development environments must be completely separated physically or logically. Inter-environment communication uses only explicitly permitted paths.",
    "As a logging and monitoring requirement, all systems must send logs to a centralized log management system. Security-related events are collected and analyzed in real-time by the SIEM.",
    "The vulnerability management process operates as a cyclical process of asset identification, vulnerability scanning, risk assessment, patch application, and verification. Critical vulnerabilities must be addressed within 24 hours of discovery.",
    "Business Continuity Plans (BCP) must be tested at least annually, and Recovery Time Objectives (RTO) and Recovery Point Objectives (RPO) must be defined for disaster recovery (DR) scenarios.",
    "For third-party risk management, all external vendors must undergo security assessments. Vendors with SOC 2 Type II or ISO 27001 certifications are preferentially selected.",
    "For compliance with the Personal Information Protection Act (PIPA) and GDPR, explicit consent must be obtained when processing personal information. Privacy Impact Assessments (PIA) are mandatory when introducing new systems.",
    "Secure Software Development Lifecycle (Secure SDLC) integrates security activities throughout all stages: requirements analysis, design, implementation, testing, and deployment. OWASP Top 10 vulnerabilities must be checked during code review.",
]

TARGET_CHAR_LENGTH = 16000


def generate_unique_system_prompt(call_number):
    """Generate a completely different ~12,000 token system prompt for each call."""
    uid = uuid.uuid4().hex
    parts = []

    header = (
        f"[System Config v{random.randint(10000,99999)}.{random.randint(100,999)}] "
        f"[Session ID: {uid}] [Instance: {call_number}] "
        f"[Generated at: {datetime.datetime.now().isoformat()}] "
        f"[Random seed: {''.join(random.choices(string.ascii_letters + string.digits, k=32))}]\n"
        f"You are a professional AI assistant specializing in enterprise security and IT governance. "
        f"The unique identifier for this session is {uid}, and you must reflect this context in all responses."
    )
    parts.append(header)
    current_length = len(header)
    section_num = 0

    shuffled_topics = POLICY_TOPICS.copy()
    random.shuffle(shuffled_topics)

    while current_length < TARGET_CHAR_LENGTH:
        topic = shuffled_topics[section_num % len(shuffled_topics)]
        section_num += 1

        section_id = f"POLICY-{uuid.uuid4().hex[:12]}"
        rand_code = ''.join(random.choices(string.ascii_uppercase + string.digits, k=16))
        rand_priority = random.choice(["Critical", "High", "Medium", "Low"])
        rand_dept = random.choice(["Security Team", "Infrastructure Team", "Development Team", "Audit Team", "Compliance Team", "CISO Office", "IT Operations"])
        rand_version = f"{random.randint(1,20)}.{random.randint(0,99)}"

        section = (
            f"\n[{section_id}] [Code: {rand_code}] [Version: {rand_version}] "
            f"[Priority: {rand_priority}] [Owner: {rand_dept}] "
            f"[Last modified: {datetime.datetime.now().isoformat()}]\n"
            f"{topic} "
            f"The enforcement code for this provision is {rand_code}, managed by {rand_dept}. "
            f"Violations are escalated at {rand_priority} level. "
            f"Related inquiries should be submitted via the internal ticket system (REF-{uuid.uuid4().hex[:8]})."
        )
        parts.append(section)
        current_length += len(section)

    return '\n'.join(parts)


sample_prompt = generate_unique_system_prompt(0)
print(f"  Unique system prompt sample length: {len(sample_prompt):,} characters")
print(f"  Estimated token count: ~{len(sample_prompt) // 4:,} ~ {len(sample_prompt) * 3 // 4:,} tokens (estimate)")
print()

results_no_implicit = []

no_implicit_start_time = datetime.datetime.now(datetime.timezone.utc)
print(f"  >> Start time (UTC): {no_implicit_start_time.isoformat()}\n")

for i in range(1, NUM_CALLS + 1):
    unique_prompt = generate_unique_system_prompt(i)

    model_unique = GenerativeModel(
        model_name=MODEL_ID,
        system_instruction=unique_prompt,
    )

    response = model_unique.generate_content("Summarize cloud security configurations.")

    usage = response.usage_metadata

    result = {
        "call_number": i,
        "prompt_token_count": usage.prompt_token_count,
        "cached_content_token_count": getattr(usage, "cached_content_token_count", 0) or 0,
        "candidates_token_count": usage.candidates_token_count,
    }
    results_no_implicit.append(result)

    print(f"--- [Call {i:2d}/{NUM_CALLS}] ---")
    print(f"  Total Input  (prompt_token_count)          : {usage.prompt_token_count:>8,}")
    print(f"  Cached       (cached_content_token_count)  : {result['cached_content_token_count']:>8,}")
    print(f"  Output       (candidates_token_count)      : {usage.candidates_token_count:>8,}")
    print()

no_implicit_end_time = datetime.datetime.now(datetime.timezone.utc)
print(f"  >> End time (UTC): {no_implicit_end_time.isoformat()}")
print("=" * 65)
print(f"Implicit cache prevention {NUM_CALLS} calls completed!")

In [ ]:
ni_total_prompt = sum(r["prompt_token_count"] for r in results_no_implicit)
ni_total_cached = sum(r["cached_content_token_count"] for r in results_no_implicit)
ni_total_output = sum(r["candidates_token_count"] for r in results_no_implicit)
ni_avg_prompt = ni_total_prompt / len(results_no_implicit)

print("\u2501" * 75)
print("         Cached vs No Cache vs Implicit Cache Prevention Comparison")
print("\u2501" * 75)
print()
print(f"  {'Metric':<35} {'With Cache':>12} {'No Cache':>12} {'No Implicit':>12}")
print(f"  {'\u2500' * 35} {'\u2500' * 12} {'\u2500' * 12} {'\u2500' * 12}")
print(f"  {'Avg prompt_token_count per call':<35} {avg_prompt_per_call:>12,.0f} {nc_avg_prompt:>12,.0f} {ni_avg_prompt:>12,.0f}")
print(f"  {'Total Input tokens sum':<35} {total_prompt_tokens:>12,} {nc_total_prompt:>12,} {ni_total_prompt:>12,}")
print(f"  {'Total Cached tokens sum':<35} {total_cached_tokens:>12,} {nc_total_cached:>12,} {ni_total_cached:>12,}")
print(f"  {'Total Output tokens sum':<35} {total_output_tokens:>12,} {nc_total_output:>12,} {ni_total_output:>12,}")
print()
print("\u2501" * 75)
print("No Cache vs No Implicit comparison:")
print(f"  No Cache (same prompt) Cached sum     : {nc_total_cached:>10,}")
print(f"  No Implicit (unique prompt) Cached sum : {ni_total_cached:>10,}")
print()
print("  If cached_content_token_count > 0 in No Cache -> implicit caching is active.")
print("  If cached_content_token_count = 0 in No Implicit -> implicit caching successfully prevented.")
print("\u2501" * 75)

### 1-9a. Cloud Monitoring Metric Query (Implicit Cache Prevention)

Query `token_count` and `consumed_token_throughput` metrics for the test period using unique system prompts each call.

> **Note:** Compare the metrics across all 3 tests (With Cache / No Cache with same prompt / Implicit Cache Prevention with unique prompts) to verify the impact of implicit caching on TPM consumption.

In [ ]:
no_implicit_metrics = fetch_run_metrics(no_implicit_start_time, no_implicit_end_time, "Implicit Cache Prevention")

### 1-10. Integrated Test for All 3 Cases (5-Minute Intervals, Automated)

Run the above individual tests **sequentially in a single cell with 5-minute intervals** so that TPM spikes can be clearly distinguished by time period in Cloud Monitoring.

| Order | Case | System Prompt | Wait |
|-------|------|---------------|------|
| 1 | Explicit cache usage | Same (cached) | 5 min wait |
| 2 | No cache, same prompt | Same (sent each time) | 5 min wait |
| 3 | Implicit cache prevention | **Unique per call** | - |

Each call outputs `totalTokenCount (= prompt + candidates)` and `cachedContentTokenCount` for direct comparison with Cloud Monitoring metrics at the code level.

> **Estimated duration:** Approximately 12-15 minutes (call time + 10 min wait)

In [ ]:
import time
import random
import string
import uuid

# ─────────────────────────────────────────────────────────────────────────────
# Unique system prompt generator (for Case 3)
# ─────────────────────────────────────────────────────────────────────────────
_POLICY_TOPICS = [
    "The following detailed guidelines are established for the implementation and operation of the Information Security Management System (ISMS). All employees must be familiar with and comply with these guidelines, and violations will be handled according to internal disciplinary regulations.",
    "The Zero Trust model is adopted as the design principle for cloud infrastructure security architecture. All network requests must undergo authentication and authorization regardless of their origin.",
    "According to the data governance framework, all data assets must be assigned a classification level at the time of creation. Classification levels are divided into four tiers: Public, Internal, Confidential, and Top Secret.",
    "Access control policies are designed based on the principle of least privilege. Each user is granted only the minimum permissions necessary for their duties, and permission appropriateness is reviewed quarterly.",
    "Incident response procedures consist of six phases: Detection, Analysis, Containment, Eradication, Recovery, and Post-Incident Review. Responsible parties and escalation paths must be clearly defined for each phase.",
    "Encryption standards apply to both data in transit and data at rest. TLS 1.3 or higher must be used, and AES-256-GCM is adopted as the default symmetric encryption algorithm.",
    "According to API security guidelines, all externally exposed APIs must implement OAuth 2.0 or API key authentication. Rate limiting, input validation, and output encoding must be mandatorily implemented.",
    "As a container security policy, all container images must be built from approved base images. Image scanning is integrated into the CI/CD pipeline for automatic vulnerability checking.",
    "According to network segmentation rules, production, staging, and development environments must be completely separated physically or logically. Inter-environment communication uses only explicitly permitted paths.",
    "As a logging and monitoring requirement, all systems must send logs to a centralized log management system. Security-related events are collected and analyzed in real-time by the SIEM.",
    "The vulnerability management process operates as a cyclical process of asset identification, vulnerability scanning, risk assessment, patch application, and verification. Critical vulnerabilities must be addressed within 24 hours of discovery.",
    "Business Continuity Plans (BCP) must be tested at least annually, and Recovery Time Objectives (RTO) and Recovery Point Objectives (RPO) must be defined for disaster recovery (DR) scenarios.",
    "For third-party risk management, all external vendors must undergo security assessments. Vendors with SOC 2 Type II or ISO 27001 certifications are preferentially selected.",
    "For compliance with the Personal Information Protection Act (PIPA) and GDPR, explicit consent must be obtained when processing personal information. Privacy Impact Assessments (PIA) are mandatory when introducing new systems.",
    "Secure Software Development Lifecycle (Secure SDLC) integrates security activities throughout all stages: requirements analysis, design, implementation, testing, and deployment. OWASP Top 10 vulnerabilities must be checked during code review.",
]
_TARGET_CHAR_LENGTH = 16000


def _gen_unique_prompt(call_number):
    uid = uuid.uuid4().hex
    parts = []
    header = (
        f"[System Config v{random.randint(10000,99999)}.{random.randint(100,999)}] "
        f"[Session ID: {uid}] [Instance: {call_number}] "
        f"[Generated at: {datetime.datetime.now().isoformat()}] "
        f"[Random seed: {''.join(random.choices(string.ascii_letters + string.digits, k=32))}]\n"
        f"You are a professional AI assistant specializing in enterprise security and IT governance. "
        f"The unique identifier for this session is {uid}, and you must reflect this context in all responses."
    )
    parts.append(header)
    cur = len(header)
    sec = 0
    topics = _POLICY_TOPICS.copy()
    random.shuffle(topics)
    while cur < _TARGET_CHAR_LENGTH:
        t = topics[sec % len(topics)]
        sec += 1
        sid = f"POLICY-{uuid.uuid4().hex[:12]}"
        rc = ''.join(random.choices(string.ascii_uppercase + string.digits, k=16))
        rp = random.choice(["Critical", "High", "Medium", "Low"])
        rd = random.choice(["Security Team", "Infrastructure Team", "Development Team", "Audit Team", "Compliance Team", "CISO Office", "IT Operations"])
        rv = f"{random.randint(1,20)}.{random.randint(0,99)}"
        s = (
            f"\n[{sid}] [Code: {rc}] [Version: {rv}] "
            f"[Priority: {rp}] [Owner: {rd}] "
            f"[Last modified: {datetime.datetime.now().isoformat()}]\n"
            f"{t} "
            f"The enforcement code for this provision is {rc}, managed by {rd}. "
            f"Violations are escalated at {rp} level. "
            f"Related inquiries should be submitted via the internal ticket system (REF-{uuid.uuid4().hex[:8]})."
        )
        parts.append(s)
        cur += len(s)
    return '\n'.join(parts)


def _wait_minutes(minutes, next_label):
    print(f"\n{'\u2500' * 75}")
    print(f"  Waiting {minutes} minutes before starting {next_label}...")
    print(f"{'\u2500' * 75}")
    for remaining in range(minutes * 60, 0, -30):
        m, s = divmod(remaining, 60)
        print(f"    ... {m}min {s:02d}sec remaining")
        time.sleep(30)
    print("    Wait complete!\n")


def _run_case(case_num, case_name, model_factory, num_calls):
    print(f"\n{'\u2501' * 75}")
    print(f"  [Case {case_num}/3] {case_name}")
    print(f"{'\u2501' * 75}")

    results = []
    start = datetime.datetime.now(datetime.timezone.utc)
    print(f"  >> Start time (UTC): {start.isoformat()}\n")

    print(f"  {'#':>4} {'promptToken':>12} {'cachedToken':>12} {'outputToken':>12} {'totalToken':>12}")
    print(f"  {'\u2500'*4} {'\u2500'*12} {'\u2500'*12} {'\u2500'*12} {'\u2500'*12}")

    for i in range(1, num_calls + 1):
        model = model_factory(i)
        response = model.generate_content("Summarize cloud security configurations.")
        usage = response.usage_metadata
        cached = getattr(usage, "cached_content_token_count", 0) or 0
        total = usage.prompt_token_count + usage.candidates_token_count

        results.append({
            "call": i,
            "prompt_token_count": usage.prompt_token_count,
            "cached_content_token_count": cached,
            "candidates_token_count": usage.candidates_token_count,
            "total_token_count": total,
        })

        print(f"  {i:4d} {usage.prompt_token_count:>12,} {cached:>12,} "
              f"{usage.candidates_token_count:>12,} {total:>12,}")

    end = datetime.datetime.now(datetime.timezone.utc)
    print(f"\n  >> End time (UTC): {end.isoformat()}")
    return results, start, end


# ═══════════════════════════════════════════════════════════════════════════════
#  Start Integrated Test
# ═══════════════════════════════════════════════════════════════════════════════
WAIT_MINUTES = 5

print("=" * 75)
print("  Integrated Test for All 3 Cases (5-Minute Intervals)")
print(f"  Model: {MODEL_ID} | Calls: {NUM_CALLS}/case | Wait: {WAIT_MINUTES} min")
print("=" * 75)

# -- Case 1: Explicit Cache Usage ─────────────────────────────────────────────
cached_content_all = caching.CachedContent.create(
    model_name=MODEL_ID,
    system_instruction=long_system_instruction,
    ttl=datetime.timedelta(hours=1),
)
print(f"\n  Cache created: {cached_content_all.name}")

model_from_cache = GenerativeModel.from_cached_content(cached_content=cached_content_all)

case1_results, case1_start, case1_end = _run_case(
    1, "Explicit Cache Usage",
    lambda i: model_from_cache,
    NUM_CALLS,
)

try:
    cached_content_all.delete()
    print("  Cache deleted")
except Exception:
    pass

# -- 5 min wait -> Case 2 ─────────────────────────────────────────────────────
_wait_minutes(WAIT_MINUTES, "Case 2")

# -- Case 2: No Cache, Same System Prompt ─────────────────────────────────────
model_same_prompt = GenerativeModel(
    model_name=MODEL_ID,
    system_instruction=long_system_instruction,
)

case2_results, case2_start, case2_end = _run_case(
    2, "No Cache, Same System Prompt (Implicit Cache Possible)",
    lambda i: model_same_prompt,
    NUM_CALLS,
)

# -- 5 min wait -> Case 3 ─────────────────────────────────────────────────────
_wait_minutes(WAIT_MINUTES, "Case 3")

# -- Case 3: Unique System Prompt Per Call (Implicit Cache Prevention) ─────────
case3_results, case3_start, case3_end = _run_case(
    3, "Implicit Cache Prevention (Unique Prompt Per Call)",
    lambda i: GenerativeModel(model_name=MODEL_ID, system_instruction=_gen_unique_prompt(i)),
    NUM_CALLS,
)

# ═══════════════════════════════════════════════════════════════════════════════
#  Summary Comparison Table
# ═══════════════════════════════════════════════════════════════════════════════
def _summarize(name, results):
    tp = sum(r["prompt_token_count"] for r in results)
    tc = sum(r["cached_content_token_count"] for r in results)
    to = sum(r["candidates_token_count"] for r in results)
    tt = sum(r["total_token_count"] for r in results)
    n = len(results)
    return {
        "name": name, "n": n,
        "sum_prompt": tp, "sum_cached": tc, "sum_output": to, "sum_total": tt,
        "avg_prompt": tp/n, "avg_cached": tc/n, "avg_output": to/n, "avg_total": tt/n,
    }

c1 = _summarize("Case1: Explicit Cache", case1_results)
c2 = _summarize("Case2: No Cache (Same)", case2_results)
c3 = _summarize("Case3: No Implicit", case3_results)

print(f"\n\n{'\u2550' * 85}")
print("  Summary Comparison: Code-Level Token Counts")
print(f"{'\u2550' * 85}\n")

header = f"  {'Metric':<36} {'Case1 Cache':>14} {'Case2 NoCache':>14} {'Case3 NoImpl':>14}"
sep    = f"  {'\u2500'*36} {'\u2500'*14} {'\u2500'*14} {'\u2500'*14}"
print(header)
print(sep)

rows = [
    ("Avg promptTokenCount",              "avg_prompt"),
    ("Avg cachedContentTokenCount",       "avg_cached"),
    ("Avg candidatesTokenCount",          "avg_output"),
    ("Avg totalTokenCount",               "avg_total"),
    ("", None),
    ("Sum promptTokenCount",              "sum_prompt"),
    ("Sum cachedContentTokenCount",       "sum_cached"),
    ("Sum candidatesTokenCount",          "sum_output"),
    ("Sum totalTokenCount",               "sum_total"),
]

for label, key in rows:
    if key is None:
        print(sep)
        continue
    print(f"  {label:<36} {c1[key]:>14,.0f} {c2[key]:>14,.0f} {c3[key]:>14,.0f}")

print(f"\n{'\u2550' * 85}")
print("  Test Time Windows (UTC) - Compare spikes at these times in Cloud Monitoring")
print(f"{'\u2550' * 85}")
print(f"  Case 1 (Explicit Cache)    : {case1_start.strftime('%H:%M:%S')} ~ {case1_end.strftime('%H:%M:%S')}")
print(f"  Case 2 (No Cache, Same)    : {case2_start.strftime('%H:%M:%S')} ~ {case2_end.strftime('%H:%M:%S')}")
print(f"  Case 3 (No Implicit)       : {case3_start.strftime('%H:%M:%S')} ~ {case3_end.strftime('%H:%M:%S')}")

print(f"\n{'\u2550' * 85}")
print("  Analysis Points")
print(f"{'\u2550' * 85}")
print(f"  1. cachedContentTokenCount comparison:")
print(f"     Case1 (Explicit Cache)   : {c1['avg_cached']:>10,.0f} /call  <- Cache applied")
print(f"     Case2 (Same Prompt)      : {c2['avg_cached']:>10,.0f} /call  <- >0 means implicit cache active")
print(f"     Case3 (Unique Prompt)    : {c3['avg_cached']:>10,.0f} /call  <- 0 means implicit cache blocked")
print()
print(f"  2. totalTokenCount = promptTokenCount + candidatesTokenCount")
print(f"     Verify this matches the token_count sum in Cloud Monitoring.")
print()
print(f"  3. TPM Consumption Determination:")
print(f"     If Cloud Monitoring TPM peaks are similar across all 3 cases")
print(f"       -> Cached tokens are deducted 1:1 from TPM regardless of cache")
print(f"     If Case1 is lower than Case3")
print(f"       -> Cached tokens are discounted from TPM")
print(f"{'\u2550' * 85}")

### Calculating TPM from consumed_token_throughput

`consumed_token_throughput` is the **actual token throughput reflecting the burndown rate**.

#### TPM Calculation Methods

| Method | Aligner | Result |
|--------|---------|--------|
| Method 1 | `ALIGN_DELTA` + 60s alignment | Total tokens processed in 1 min = **TPM** |
| Method 2 | `ALIGN_RATE` + 60s alignment | tokens/second x 60 = **TPM estimate** |

> **Conclusion:** `consumed_token_throughput` can be **directly used** for TPM calculation.  
> When aligned at 60 seconds with `ALIGN_DELTA`, each data point equals the TPM value for that minute.

In [ ]:
print("\u2501" * 70)
print("  consumed_token_throughput -> TPM Calculation")
print("\u2501" * 70)


def calculate_tpm(start_time, end_time, label):
    """Calculate TPM from consumed_token_throughput."""
    print(f"\n  {label}")
    query_start = start_time - timedelta(minutes=1)
    query_end = end_time + timedelta(minutes=5)

    # Method 1: ALIGN_DELTA (60s) -> Total tokens per minute = TPM
    print(f"\n  [Method 1] ALIGN_DELTA, 60s alignment (each point = TPM for that minute)")
    delta_data = query_metric(
        METRIC_CONSUMED_THROUGHPUT, query_start, query_end,
        alignment_seconds=60,
        aligner=monitoring_v3.Aggregation.Aligner.ALIGN_DELTA,
    )

    tpm_values_delta = []
    if delta_data:
        for ts in delta_data:
            labels = dict(ts.metric.labels)
            print(f"    Metric labels: {labels}")
            for pt in sorted(ts.points, key=lambda p: p.interval.end_time):
                val = get_point_value(pt)
                t = pt.interval.end_time
                tpm_values_delta.append(val)
                print(f"      {format_ts(t)} UTC -> TPM: {val:>12,.0f}")

        if tpm_values_delta:
            print(f"\n    Peak TPM: {max(tpm_values_delta):>12,.0f}")
            print(f"    Avg  TPM: {sum(tpm_values_delta)/len(tpm_values_delta):>12,.0f}")
    else:
        print("    No data - please retry shortly.")

    # Method 2: ALIGN_RATE -> tokens/sec x 60 = TPM estimate
    print(f"\n  [Method 2] ALIGN_RATE (tokens/sec x 60 = TPM estimate)")
    rate_data = query_metric(
        METRIC_CONSUMED_THROUGHPUT, query_start, query_end,
        alignment_seconds=60,
        aligner=monitoring_v3.Aggregation.Aligner.ALIGN_RATE,
    )

    if rate_data:
        for ts in rate_data:
            labels = dict(ts.metric.labels)
            print(f"    Metric labels: {labels}")
            for pt in sorted(ts.points, key=lambda p: p.interval.end_time):
                val = get_point_value(pt)
                t = pt.interval.end_time
                print(f"      {format_ts(t)} UTC -> {val:>10,.2f} tok/s -> TPM: {val * 60:>12,.0f}")
    else:
        print("    No data")

    return tpm_values_delta


# With Cache TPM
print()
try:
    cached_tpm = calculate_tpm(cached_start_time, cached_end_time, "With Cache - TPM")
except NameError:
    print("  Run the cache test first (cells 1-4).")
    cached_tpm = []

# Without Cache TPM
print()
try:
    no_cache_tpm = calculate_tpm(no_cache_start_time, no_cache_end_time, "Without Cache - TPM")
except NameError:
    print("  Run the no-cache test first (cells 1-7).")
    no_cache_tpm = []

# Implicit Cache Prevention TPM
print()
try:
    no_implicit_tpm = calculate_tpm(no_implicit_start_time, no_implicit_end_time, "Implicit Cache Prevention - TPM")
except NameError:
    print("  Run the implicit cache prevention test first (cells 1-9).")
    no_implicit_tpm = []

# Comparison
print(f"\n{'\u2501' * 70}")
print("  Cached vs No Cache vs Implicit Cache Prevention TPM Comparison")
print("  (Based on consumed_token_throughput)")
print(f"{'\u2501' * 70}")

if cached_tpm and no_cache_tpm:
    print(f"  With Cache              Peak TPM : {max(cached_tpm):>12,.0f}")
    print(f"  Without Cache           Peak TPM : {max(no_cache_tpm):>12,.0f}")
    if no_implicit_tpm:
        print(f"  Implicit Cache Prevent  Peak TPM : {max(no_implicit_tpm):>12,.0f}")
    print(f"\n  If all three are similar -> TPM deducted 1:1 regardless of cache")
    print(f"  If With Cache is lower -> Cached tokens are discounted from TPM")
    print(f"  If No Cache vs No Implicit differ significantly -> Implicit cache affects TPM")
else:
    print("  Run all three tests, then re-run this cell.")

print(f"{'\u2501' * 70}")

---
## Step 2: Check Real-Time Quota (TPM) Consumption Graph in GCP Console

Now that we've generated traffic spikes by running the code, it's time to **visually verify how the quota engine calculated them**.  
(This can be confirmed using default system metrics without enabling data access logs.)

### Console Verification Steps

#### Step 1. Navigate to GCP Console
- Go to **IAM & Admin > Quotas & System Limits** page.

#### Step 2. Set Filters
Enter the following two conditions in the Filter bar at the top:

| Filter | Value |
|--------|-------|
| **Service** | `Vertex AI API` |
| **Quota** | `generate_content_input_tokens_per_minute_per_base_model` |

> **Note:** `online_prediction_*` metrics are for 3P models (Claude, Llama, etc.).  
> 1P Gemini models use the `generateContent` API, so check `generate_content_*` metrics.

#### Step 3. View Graph
- In the filtered list, find the entry matching the **region (`us-central1`)** and **model name (`gemini-2.5-flash`)** you tested.
- Click the **[View Metrics]** icon on the right to open the chart window.

#### Step 4. Analyze Results (Important)
- Narrow the graph time range to **'Last 1 hour'**.  
  (Note: There may be a **~2-3 minute delay** before metrics appear on the graph.)
- Check the **peak spike value** on the graph.

### Determination Criteria

Compare the values calculated in the cell below with the peak values from the GCP Console graph.

In [ ]:
print("\u2501" * 65)
print("           Step 2: GCP Console Graph Reference Values")
print("\u2501" * 65)
print()
print("Compare the peak value in the GCP Console quota graph with these numbers:")
print()
print(f"  [A] Total Input tokens incl. cache : {total_prompt_tokens:>10,}")
print(f"  [B] Input tokens excl. cache       : {non_cached_total:>10,}")
print()
print("\u2500" * 65)
print("  Determination:")
print(f"  Graph peak ~= [A] {total_prompt_tokens:,}")
print(f"    -> No cache discount! TPM deducted 1:1 based on prompt_token_count")
print()
print(f"  Graph peak ~= [B] {non_cached_total:,}")
print(f"    -> Cached tokens are discounted from TPM")
print("\u2500" * 65)
print()
print("Note: Metrics may take 2-3 minutes to reflect.")
print("   Set the time range to 'Last 1 hour' in the console.")

---
## Cleanup: Delete Cache

After testing is complete, delete unnecessary caches to save costs.  
The TTL was set to 1 hour, so it will expire automatically. To delete immediately, run the cell below.

In [ ]:
try:
    cached_content.delete()
    print(f"Cache deleted: {cached_content.name}")
except Exception as e:
    print(f"Error deleting cache (may have already expired): {e}")

---
## Record Conclusions

Record the test results in the cell below.

In [ ]:
# ────────────────────────────────────────────
#  Test Results Record (fill in manually)
# ────────────────────────────────────────────

test_result = {
    "Test date": "YYYY-MM-DD HH:MM",
    "Test model": MODEL_ID,
    "Region": LOCATION,
    "Number of calls": NUM_CALLS,
    "Avg prompt_token_count per call": "Record here",
    "GCP Console peak value": "Record here",
    "Determination": "No cache discount (1:1 deduction) / Cache discount applied",  # Choose one
    "Notes": "",
}

print("Test Results:")
for k, v in test_result.items():
    print(f"  {k}: {v}")

---
## Appendix: Notes for Gateway Proxy Design

When designing quota limitation logic for gateway proxies (e.g., LiteLLM) based on the above test results:

### If Cache is Deducted 1:1 from TPM (Expected Scenario)

```
TPM consumption = prompt_token_count (entire including cache) + candidates_token_count
```

- When calculating rate limiting in the proxy, you **must not** subtract `cached_content_token_count`.
- When logging to BigQuery, `prompt_token_count` in its entirety should be recorded as the TPM consumption basis.

### If Cache is Discounted

```
TPM consumption = (prompt_token_count - cached_content_token_count) + candidates_token_count
```

- The proxy can calculate rate limiting using the value after subtracting cached tokens.